# NBA Streamer Prediction Model

In [1]:
!brew install libomp
!pip install scikit-learn

==> Downloading Homebrew API data
⠋ JSON API packages.arm64_tahoe.jws.json            Downloading  15.7MB/-------⠋ JSON API packages.arm64_tahoe.jws.json            Downloading  15.7MB/-------⠙ JSON API packages.arm64_tahoe.jws.json            Downloading  15.7MB/-------⠚ JSON API packages.arm64_tahoe.jws.json            Downloading  15.7MB/-------⠞ JSON API packages.arm64_tahoe.jws.json            Downloading  15.7MB/-------✔︎ JSON API packages.arm64_tahoe.jws.json            Downloaded   15.7MB/ 15.7MB
To reinstall 22.1.8, run:
  brew reinstall libomp
  Using cached scikit_learn-1.9.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp314-cp314-macosx_12_0_arm64.whl (8.2 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 72.2 MB/s  0:00:00 eta 0:00:01
Using ca

In [2]:
import os
import pandas as pd
import lightgbm as lgb
from scipy import stats
import numpy as np

In [3]:
# Define directory with processed datasets
PROCESSED_DIRECTORY = os.path.join("nba_training_data", "processed")

# (1) Import the data

In [4]:
def load_data():
        # Define file paths
        train_path = os.path.join(PROCESSED_DIRECTORY, "train_streamers.csv")
        val_path = os.path.join(PROCESSED_DIRECTORY, "val_streamers.csv")
        test_path = os.path.join(PROCESSED_DIRECTORY, "test_streamers.csv")

        if not os.path.exists(train_path):
                raise FileNotFoundError("Processed files not found. Run the preprocessing script first.")

        # Import the data
        train_df = pd.read_csv(train_path, parse_dates=['GAME_DATE'])
        val_df = pd.read_csv(val_path, parse_dates=['GAME_DATE'])
        test_df = pd.read_csv(test_path, parse_dates=['GAME_DATE'])

        # Sort the datasets
        train_df = train_df.sort_values(['GAME_DATE', 'TEAM_ID', 'PLAYER_ID']).reset_index(drop=True)
        val_df = val_df.sort_values(['GAME_DATE', 'TEAM_ID', 'PLAYER_ID']).reset_index(drop=True)
        test_df = test_df.sort_values(['GAME_DATE', 'TEAM_ID', 'PLAYER_ID']).reset_index(drop=True)

        return train_df, val_df, test_df


In [5]:
train_df, val_df, test_df = load_data()

# (2) Feature selection

In [6]:
features = [
        # Baseline momentum and overall FP generation/averages and min
        'FP_ROLLING_3', 'FP_ROLLING_4', 'FP_ROLLING_10', 'FP_PER_MIN_10',
        'MIN_ROLLING_3', 'MIN_ROLLING_4', 'MIN_ROLLING_10', 
        
        # Player stat profiles
        'PTS_ROLLING_4', 'REB_ROLLING_4', 'AST_ROLLING_4',
        'STL_ROLLING_4', 'BLK_ROLLING_4', 'TOV_ROLLING_4', 'FG3M_ROLLING_4',
        
        # The vacated stats/opportunities (Identifying injuries/rest)
        'TEAM_VACATED_MINUTES', 'TEAM_VACATED_PTS', 
        'TEAM_VACATED_REB', 'TEAM_VACATED_AST',

        # Opponnent defensive rating and pace
        'OPP_DEF_RTG_10', 'OPP_PACE_10',
        
        'HOME_GAME'
]

# (3) Training Predictor Model

In [7]:
# Training a LGBMRegressor model to predict the Delta (Points above/below the
# 10-game rolling average), using MAE
def train_model(train_df, val_df, features):
        X_train = train_df[features]
        y_train = train_df['FANTASY_PTS'] - train_df['FP_ROLLING_10']

        X_val = val_df[features]
        y_val = val_df['FANTASY_PTS'] - val_df['FP_ROLLING_10']

        regressor = lgb.LGBMRegressor(
                objective="mae",
                learning_rate=0.03,
                n_estimators=3000,
                num_leaves=63,
                max_depth=7,
                min_child_samples=30,
                random_state=42
        )

        regressor.fit(
                X_train, y_train,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(stopping_rounds=30)]
        )

        return regressor


In [8]:
print("Training LGBMRegressor...")
model = train_model(train_df, val_df, features)

Training LGBMRegressor...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000936 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3481
[LightGBM] [Info] Number of data points in the train set: 64967, number of used features: 21
[LightGBM] [Info] Start training from score -0.777778
Training until validation scores don't improve for 30 rounds


/Users/carterng-yu/Desktop/NBA_Fantasy_Basketball_Project/NBA_STREAMER_ML_MODEL/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [9]:
# We can see which features the model deems the most important when making its
# predictions.
print("\n--- Feature Importances ---")
importance = pd.DataFrame({
        'Feature': features,
        'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)
print(importance.to_string(index=False))


--- Feature Importances ---
             Feature  Importance
       FP_ROLLING_10        1260
       MIN_ROLLING_3         955
       FP_PER_MIN_10         894
      OPP_DEF_RTG_10         872
      MIN_ROLLING_10         784
         OPP_PACE_10         715
        FP_ROLLING_3         641
       REB_ROLLING_4         627
       PTS_ROLLING_4         537
    TEAM_VACATED_PTS         532
    TEAM_VACATED_REB         528
       AST_ROLLING_4         495
       TOV_ROLLING_4         493
TEAM_VACATED_MINUTES         464
       MIN_ROLLING_4         440
        FP_ROLLING_4         417
    TEAM_VACATED_AST         385
      FG3M_ROLLING_4         363
       STL_ROLLING_4         231
       BLK_ROLLING_4         229
           HOME_GAME         119


It seems that FP_ROLLING_10 is the most important and this makes sense as it is
closely related to what the model is trying to predict. MIN_ROLLING_3 is
important because it represents the recent opportunity/play-time that a player
is receiving. FP_PER_MIN_10 demonstrates the efficiency of a player. Finally,
OPP_DEF_RTG_10 was deemed important due to a player having a higher chance to
score and generate fantasy points when their opponent is worse at defense.

# (4) Model Testing and Evaluation

## (4.1) The Eye Test

This test simulates a fantasy season by comparing the ML model's prediction for
the best streamer of the day against a Baseline pick of the player with the
highest 10-game average FP. The model's predicted FP, predict delta, and actual
FP scores of the players is outputted. The Baseline's player FP average and
their actual score that day are also outputted.

In [10]:
def simulate_season(test_df, model, features):
        print("\n--- Running Daily Streamer Simulation (Test Season) ---")

        test_df = test_df.copy()
    
        # Store predicted delta and final projected fantasy points
        test_df['PREDICTED_DELTA'] = model.predict(test_df[features])
        test_df['ML_PREDICTION'] = test_df['FP_ROLLING_10'] + test_df['PREDICTED_DELTA']

        daily_results = []

        # Iterate through every day, printing the model's and baseline's 
        # picks and important data for each daily matchup
        for date, slate in test_df.groupby('GAME_DATE'):
                if len(slate) < 2:
                        continue

                baseline_pick = slate.sort_values(by='FP_ROLLING_10', ascending=False).iloc[0]
                ml_pick = slate.sort_values(by='ML_PREDICTION', ascending=False).iloc[0]

                base_name = baseline_pick.get('PLAYER_NAME', baseline_pick['PLAYER_ID'])
                ml_name = ml_pick.get('PLAYER_NAME', ml_pick['PLAYER_ID'])
                
                base_actual = baseline_pick['FANTASY_PTS']
                ml_actual = ml_pick['FANTASY_PTS']
                daily_diff = ml_actual - base_actual
                
                # Predicted metrics & confidence
                ml_proj = ml_pick['ML_PREDICTION']
                ml_delta = ml_pick['PREDICTED_DELTA']
                base_proj = baseline_pick['FP_ROLLING_10']
                
                # Projected edge over the baseline pick on this slate
                proj_edge = ml_proj - baseline_pick['ML_PREDICTION']

                # Format date cleanly
                date_str = date.strftime('%Y-%m-%d') if hasattr(date, 'strftime') else str(date)

                # Print matchup with model confidence breakdown
                if ml_name == base_name:
                        print(
                        f"[{date_str}] TIE: Both picked {ml_name} "
                        f"| Proj: {ml_proj:.1f} (Δ: {ml_delta:+.1f}) | Actual: {ml_actual:.1f} pts"
                )
                else:
                        print(
                        f"[{date_str}] ML: {ml_name} [Proj: {ml_proj:.1f}, Δ: {ml_delta:+.1f}, Actual: {ml_actual:.1f}] "
                        f"vs BASE: {base_name} [Base Avg: {base_proj:.1f}, Actual: {base_actual:.1f}] "
                        f"| Proj Edge: {proj_edge:+.1f} | Real Edge: {daily_diff:+.1f} pts"
                        )

                daily_results.append({
                        'GAME_DATE': date,
                        'BASELINE_PLAYER_ID': baseline_pick['PLAYER_ID'],
                        'ML_PLAYER_ID': ml_pick['PLAYER_ID'],
                        'BASELINE_FP': base_actual,
                        'ML_FP': ml_actual,
                        'ML_PREDICTED_DELTA': ml_delta,
                        'ML_PREDICTED_TOTAL': ml_proj,
                        'PROJ_EDGE_OVER_BASELINE': proj_edge
                })

        results_df = pd.DataFrame(daily_results)
        results_df['DIFF'] = results_df['ML_FP'] - results_df['BASELINE_FP']
        results_df['SAME_PICK'] = results_df['ML_PLAYER_ID'] == results_df['BASELINE_PLAYER_ID']

        total_slates = len(results_df)
        baseline_total_pts = results_df['BASELINE_FP'].sum()
        ml_total_pts = results_df['ML_FP'].sum()

        # Print the final results of the test season simulation (2025-26)
        print(f"\nTotal Slates Simulated: {total_slates}")
        print(f"Baseline Manager Total Points: {baseline_total_pts:.1f}")
        print(f"ML Manager Total Points:       {ml_total_pts:.1f}")

        diff = ml_total_pts - baseline_total_pts
        if diff > 0:
                print(f"\nResult: The ML Model BEAT the Baseline by {diff:.1f} fantasy points!")
                print(f"That is an average advantage of {diff/total_slates:.2f} points per day.")
        else:
                print(f"\nResult: The Baseline BEAT the ML Model by {abs(diff):.1f} fantasy points!")

        return results_df

In [11]:
results_df = simulate_season(test_df, model, features)


--- Running Daily Streamer Simulation (Test Season) ---
[2025-10-21] ML: Jabari Smith Jr. [Proj: 29.5, Δ: +4.2, Actual: 26.0] vs BASE: Aaron Wiggins [Base Avg: 27.7, Actual: 13.0] | Proj Edge: +0.3 | Real Edge: +13.0 pts
[2025-10-22] ML: Jonathan Mogbo [Proj: 34.0, Δ: +1.2, Actual: 8.0] vs BASE: Tre Jones [Base Avg: 38.6, Actual: 39.0] | Proj Edge: +3.5 | Real Edge: -31.0 pts
[2025-10-23] TIE: Both picked Aaron Wiggins | Proj: 28.2 (Δ: +1.4) | Actual: 39.0 pts
[2025-10-24] TIE: Both picked Davion Mitchell | Proj: 31.2 (Δ: -6.1) | Actual: 9.0 pts
[2025-10-25] TIE: Both picked VJ Edgecombe | Proj: 37.3 (Δ: -11.7) | Actual: 47.0 pts
[2025-10-26] TIE: Both picked Davion Mitchell | Proj: 29.4 (Δ: -5.0) | Actual: 23.0 pts
[2025-10-27] TIE: Both picked VJ Edgecombe | Proj: 43.3 (Δ: -4.7) | Actual: 53.0 pts
[2025-10-28] TIE: Both picked VJ Edgecombe | Proj: 42.4 (Δ: -7.3) | Actual: 28.0 pts
[2025-10-29] TIE: Both picked Tre Jones | Proj: 31.2 (Δ: -11.6) | Actual: 29.0 pts
[2025-10-30] TIE: Bo

To the eye, the test season simulation seems to be successful as the model beat
the Baseline by a significant amount of points (195), but fantasy basketball is 
inherently noisy, and an average of +1.19 points per day may be just noise. 
Because of this, I have decided to undergo additional testing and evaluation to
determine if the model has a significant advantage of the Baseline and whether
the model performs better in different scenarios (weekly picks, total rankings).

## (4.2) Statistical Significance

In [12]:
# Test to determine if the model's 1.19 point edge is significant or just noise
# Compares the point differences (ML vs. Baseline) and outputs sample size, mean
# point edge, standard deviation, model win rate, effect size (Cohen's d), 
# p-values for both a parametric t-test and a non-parametric Wilcoxon test, and 
# a simulated 95% Bootstrap Confidence Interval.
def _paired_summary(diffs, label, n_bootstrap=10000, random_state=42):
        n = len(diffs)
        if n == 0:
                print(f"{label}: no slates to test.")
                return None

        mean_diff = diffs.mean()
        std_diff = diffs.std(ddof=1) if n > 1 else float('nan')

        t_stat, t_pval = stats.ttest_1samp(diffs, popmean=0) if n > 1 else (float('nan'), float('nan'))
        try:
                w_stat, w_pval = stats.wilcoxon(diffs)
        except ValueError:
                w_stat, w_pval = float('nan'), float('nan')

        rng = np.random.default_rng(random_state)
        boot_means = np.array([
                rng.choice(diffs, size=n, replace=True).mean()
                for _ in range(n_bootstrap)
        ]) if n > 1 else np.array([mean_diff])
        ci_low, ci_high = np.percentile(boot_means, [2.5, 97.5])

        win_rate = (diffs > 0).mean()
        cohens_d = mean_diff / std_diff if std_diff and std_diff > 0 else float('nan')

        print(f"\n{label}  (n = {n})")
        print(f"  Mean edge:            {mean_diff:.2f} pts  (std: {std_diff:.2f})")
        print(f"  ML win rate:          {win_rate:.1%}")
        print(f"  Cohen's d:            {cohens_d:.3f}")
        print(f"  Paired t-test:        t = {t_stat:.3f}, p = {t_pval:.4f}")
        print(f"  Wilcoxon signed-rank: W = {w_stat:.1f}, p = {w_pval:.4f}")
        print(f"  Bootstrap 95% CI:     [{ci_low:.2f}, {ci_high:.2f}] pts")

        return {'n': n, 'mean_diff': mean_diff, 't_pval': t_pval, 'w_pval': w_pval,
                'ci_low': ci_low, 'ci_high': ci_high, 'win_rate': win_rate, 'cohens_d': cohens_d}

In [13]:
# Uses the helper function above (_paired_summary) and determines if the model's
# point edge is significant or noise. It filters out the days when the Model
# and the Baseline choose the same player, as those days are uninformative and
# would hinder the overall test. Using the disagreement-only days isolates
# the comparisons worth testing.
def run_significance_test(results_df, n_bootstrap=10000, random_state=42):
        same_pick_rate = results_df['SAME_PICK'].mean()
        print("--- Significance Check: ML edge over Baseline ---")
        print(f"Same player picked by both methods on {same_pick_rate:.1%} of slates "
                f"(these days are uninformative ties by construction).")

        overall = _paired_summary(results_df['DIFF'].to_numpy(), 
                                  "Overall (all slates)",
                                  n_bootstrap=n_bootstrap, 
                                  random_state=random_state)

        disagreement_df = results_df[~results_df['SAME_PICK']]
        disagreement = _paired_summary(disagreement_df['DIFF'].to_numpy(),
                                       "Disagreement-only (methods picked different players)",
                                       n_bootstrap=n_bootstrap, 
                                       random_state=random_state)

        if disagreement is not None:
                ci_low, ci_high = disagreement['ci_low'], disagreement['ci_high']
                print()
                if ci_low > 0:
                        print("-> On days the model disagrees with the baseline, its pick genuinely "
                              "outperforms: 95% CI is entirely above 0.")
                elif ci_high < 0:
                        print("-> On days the model disagrees with the baseline, its pick underperforms: "
                              "95% CI is entirely below 0. The model's disagreements are net negative.")
                else:
                        print("-> Even isolating to disagreement days, the CI still straddles 0: "
                              "not enough evidence yet that the model's differentiated picks add value.")

        return {'overall': overall, 'disagreement_only': disagreement, 'same_pick_rate': same_pick_rate}


In [14]:
run_significance_test(results_df)
print()

--- Significance Check: ML edge over Baseline ---
Same player picked by both methods on 29.3% of slates (these days are uninformative ties by construction).

Overall (all slates)  (n = 164)
  Mean edge:            1.19 pts  (std: 15.50)
  ML win rate:          39.0%
  Cohen's d:            0.077
  Paired t-test:        t = 0.983, p = 0.3273
  Wilcoxon signed-rank: W = 2947.5, p = 0.2794
  Bootstrap 95% CI:     [-1.18, 3.64] pts

Disagreement-only (methods picked different players)  (n = 116)
  Mean edge:            1.68 pts  (std: 18.43)
  ML win rate:          55.2%
  Cohen's d:            0.091
  Paired t-test:        t = 0.983, p = 0.3279
  Wilcoxon signed-rank: W = 2947.5, p = 0.2794
  Bootstrap 95% CI:     [-1.64, 5.02] pts

-> Even isolating to disagreement days, the CI still straddles 0: not enough evidence yet that the model's differentiated picks add value.



This significance check shows that in all the slates, the two methods tie 29.3% 
of the time, the model wins 39% of the time, and the baseline wins the remaining
31.7% of the time. Overall, it is more important to look only at the days where
the two methods disagree, and the model wins 55.2% of these slates with a 1.68
mean edge. Cohen's d represents the effect size, and 0.091 reveals that the 
effect is small. The p-value represents the chance of an entirely random model
producing the same results, with 0.32 being a medium chance (the smaller the 
p-value the better). The Wilcoxon test reiterates this point, and the final 
Bootstrap test reveals that in 95% of realistic situations, the model will range
from losing by 1.64 points to the baseline, to winning by 5.02 points. This test
reveals that there is no concrete proof so far that the model is stastically 
superior than the baseline at choosing the best streamer for a given day. 
HOWEVER, this is merely one application of the model that may not even be the 
most useful way for the model to be used. 
        Below, I continue testing and find that the MODEL HAS STATISTICAL
SIGNIFICANCE in other important and realistic scenarios, such as predicting the 
best streamer of a given week.



## (4.3) Weekly Prediction

In [16]:
# Simulate a test season week-by-week, printing the Model's choice vs. the
# Baseline's choice. Also prints out a significance test for the weekly head to
# head matchups on the disagreement only weeks, a significance test for the
# Model's prediction of every player's total points for the week, and a
# significance test for the Model's ranking of the entire streamer pool for a
# given week in comparison to the Baseline.
def evaluate_weekly_totals(test_df, model, features, min_players_per_week=5):
        test_df = test_df.copy()

        # Store predicted delta and final projected fantasy points
        test_df['PREDICTED_DELTA'] = model.predict(test_df[features])
        test_df['ML_PREDICTION'] = test_df['FP_ROLLING_10'] + test_df['PREDICTED_DELTA']

        iso = test_df['GAME_DATE'].dt.isocalendar()
        test_df['ISO_YEAR'] = iso['year']
        test_df['ISO_WEEK'] = iso['week']

        # Include PLAYER_NAME if available for clean printouts
        group_cols = ['PLAYER_ID', 'ISO_YEAR', 'ISO_WEEK']
        has_name = 'PLAYER_NAME' in test_df.columns
        if has_name:
                group_cols.append('PLAYER_NAME')

        weekly = test_df.groupby(group_cols).agg(
                GAMES_PLAYED=('FANTASY_PTS', 'size'),
                ACTUAL_WEEKLY_FP=('FANTASY_PTS', 'sum'),
                ML_WEEKLY_FP=('ML_PREDICTION', 'sum'),
                BASELINE_WEEKLY_FP=('FP_ROLLING_10', 'sum'),
        ).reset_index()

         # --- Head-to-Head Weekly Matchups Printout ---
        print("\n--- Running Weekly Streamer Matchups ---")
        weekly_corrs = []
        weekly_picks = []  # per-week pick outcomes, for the disagreement-only significance test

        baseline_total_pts = 0
        ml_total_pts = 0
        weeks_simulated = 0

        for (year, wk), group in weekly.groupby(['ISO_YEAR', 'ISO_WEEK']):
                if len(group) < min_players_per_week:
                        continue

                # Identify the #1 pick for the week by each method
                baseline_pick = group.sort_values(by='BASELINE_WEEKLY_FP', ascending=False).iloc[0]
                ml_pick = group.sort_values(by='ML_WEEKLY_FP', ascending=False).iloc[0]

                base_name = baseline_pick['PLAYER_NAME'] if has_name else baseline_pick['PLAYER_ID']
                ml_name = ml_pick['PLAYER_NAME'] if has_name else ml_pick['PLAYER_ID']

                base_proj = baseline_pick['BASELINE_WEEKLY_FP']
                base_actual = baseline_pick['ACTUAL_WEEKLY_FP']

                ml_proj = ml_pick['ML_WEEKLY_FP']
                ml_actual = ml_pick['ACTUAL_WEEKLY_FP']

                # Calculate Delta for the week (ML Proj - Baseline Proj for that player)
                ml_delta = ml_proj - ml_pick['BASELINE_WEEKLY_FP']
                proj_edge = ml_proj - baseline_pick['ML_WEEKLY_FP']
                week_diff = ml_actual - base_actual

                # PLAYER_ID (not name) determines agreement -- robust to any
                # duplicate/missing PLAYER_NAME values.
                same_pick = ml_pick['PLAYER_ID'] == baseline_pick['PLAYER_ID']

                baseline_total_pts += base_actual
                ml_total_pts += ml_actual
                weeks_simulated += 1

                weekly_picks.append({
                        'ISO_YEAR': year, 'ISO_WEEK': wk,
                        'ML_PLAYER_ID': ml_pick['PLAYER_ID'], 'BASELINE_PLAYER_ID': 
                        baseline_pick['PLAYER_ID'], 'ML_ACTUAL': ml_actual, 
                        'BASELINE_ACTUAL': base_actual, 'DIFF': week_diff, 
                        'SAME_PICK': same_pick,
                })

                # Print matchup with model confidence breakdown
                if same_pick:
                        print(f"[Year {year} | Wk {wk:02d}] TIE: Both picked {ml_name} "
                        f"| Proj: {ml_proj:.1f} (Δ: {ml_delta:+.1f}) | Actual: {ml_actual:.1f} pts"
                        )
                else:
                        print(
                                f"[Year {year} | Wk {wk:02d}] ML: {ml_name} [Proj: {ml_proj:.1f}, Δ: {ml_delta:+.1f}, Actual: {ml_actual:.1f}] "
                                f"vs BASE: {base_name} [Base Avg: {base_proj:.1f}, Actual: {base_actual:.1f}] "
                                f"| Proj Edge: {proj_edge:+.1f} | Real Edge: {week_diff:+.1f} pts"
                        )

                # --- Weekly ranking quality (uses the full slate, unaffected by this change) ---
                ml_corr, _ = stats.spearmanr(group['ML_WEEKLY_FP'], group['ACTUAL_WEEKLY_FP'])
                baseline_corr, _ = stats.spearmanr(group['BASELINE_WEEKLY_FP'], group['ACTUAL_WEEKLY_FP'])

                if np.isnan(ml_corr) or np.isnan(baseline_corr):
                        continue

                weekly_corrs.append({'ISO_YEAR': year, 'ISO_WEEK': wk, 'N_PLAYERS': len(group),
                                'ML_CORR': ml_corr, 'BASELINE_CORR': baseline_corr})

        # --- Matchup Summary (descriptive totals, all weeks) ---
        print(f"\n--- Season Summary (Weekly Picks) ---")
        print(f"Total Weeks Simulated: {weeks_simulated}")
        print(f"Baseline Manager Total Points: {baseline_total_pts:.1f}")
        print(f"ML Manager Total Points:       {ml_total_pts:.1f}")

        diff = ml_total_pts - baseline_total_pts
        if diff > 0:
                print(f"\nResult: The ML Model BEAT the Baseline by {diff:.1f} fantasy points!")
        elif diff < 0:
                print(f"\nResult: The Baseline BEAT the ML Model by {abs(diff):.1f} fantasy points!")
        else:
                print("\nResult: TIE.")

        # --- Head-to-head significance: DISAGREEMENT WEEKS ONLY ---
        weekly_picks_df = pd.DataFrame(weekly_picks)
        same_pick_rate = weekly_picks_df['SAME_PICK'].mean() if len(weekly_picks_df) else float('nan')
        print(f"\nSame player picked by both methods on {same_pick_rate:.1%} of weeks "
              f"(these weeks are uninformative ties by construction).")

        disagreement_weeks = weekly_picks_df[~weekly_picks_df['SAME_PICK']]
        _paired_summary(disagreement_weeks['DIFF'].to_numpy(),
                     "Weekly manager pick, disagreement-only (ML actual - Baseline actual)")

        # --- Weekly total accuracy (unchanged -- uses ALL player-weeks, not just picks) ---
        weekly['ML_ERROR'] = weekly['ACTUAL_WEEKLY_FP'] - weekly['ML_WEEKLY_FP']
        weekly['BASELINE_ERROR'] = weekly['ACTUAL_WEEKLY_FP'] - weekly['BASELINE_WEEKLY_FP']

        print("\n--- Weekly Total Accuracy (player-weeks, test season) ---")
        print(f"Player-weeks evaluated: {len(weekly)}")

        accuracy_improvement = weekly['BASELINE_ERROR'].abs() - weekly['ML_ERROR'].abs()
        _paired_summary(accuracy_improvement.to_numpy(),
                        "Weekly total accuracy: |Baseline error| - |ML error| per player-week")

        # --- Output Final Ranking Stats ---
        weekly_corr_df = pd.DataFrame(weekly_corrs)
        weekly_corr_df['CORR_DIFF'] = weekly_corr_df['ML_CORR'] - weekly_corr_df['BASELINE_CORR']

        print(f"\n--- Weekly Ranking Quality (Spearman: 'who's the best streamer this week') ---")
        print(f"Weeks evaluated: {len(weekly_corr_df)} (weeks with >= {min_players_per_week} active players)")
        print(f"Mean ML weekly correlation:       {weekly_corr_df['ML_CORR'].mean():.4f}")
        print(f"Mean Baseline weekly correlation: {weekly_corr_df['BASELINE_CORR'].mean():.4f}")

        _paired_summary(weekly_corr_df['CORR_DIFF'].to_numpy(), "ML vs Baseline weekly ranking correlation")

        return weekly, weekly_corr_df, weekly_picks_df


In [17]:
weekly_df, weekly_corr_df, weekly_picks_df = evaluate_weekly_totals(test_df, model, features)


--- Running Weekly Streamer Matchups ---
[Year 2025 | Wk 43] TIE: Both picked Davion Mitchell | Proj: 91.2 (Δ: -15.5) | Actual: 89.0 pts
[Year 2025 | Wk 44] TIE: Both picked VJ Edgecombe | Proj: 157.9 (Δ: -26.4) | Actual: 148.0 pts
[Year 2025 | Wk 45] TIE: Both picked VJ Edgecombe | Proj: 139.2 (Δ: -11.4) | Actual: 84.0 pts
[Year 2025 | Wk 46] TIE: Both picked Isaiah Collier | Proj: 118.1 (Δ: +0.8) | Actual: 132.0 pts
[Year 2025 | Wk 47] TIE: Both picked Brandon Williams | Proj: 100.2 (Δ: -1.3) | Actual: 95.0 pts
[Year 2025 | Wk 48] ML: Royce O'Neale [Proj: 93.0, Δ: -0.9, Actual: 116.0] vs BASE: Cedric Coward [Base Avg: 97.1, Actual: 58.0] | Proj Edge: +9.1 | Real Edge: +58.0 pts
[Year 2025 | Wk 49] TIE: Both picked Kevin Porter Jr. | Proj: 135.5 (Δ: -29.0) | Actual: 224.0 pts
[Year 2025 | Wk 50] TIE: Both picked Kevin Porter Jr. | Proj: 79.0 (Δ: -16.5) | Actual: 71.0 pts
[Year 2025 | Wk 51] TIE: Both picked Jaime Jaquez Jr. | Proj: 105.1 (Δ: +5.4) | Actual: 101.0 pts
[Year 2025 | Wk 

This output reveals that the MODEL HAS STATISTICAL SIGNIFICANCE when predicting
the best streamer of the week compared to the Baseline. Over a 25 week test
season, the Model BEAT the Baseline by 209 points, AVERAGING +8.36 points per
matchup. On the disagreement only slates, the model WINS 66.7% of the time with
an AVERAGE of +17.42 points. Cohen's d revelas this to be a medium effect with
the p-values and the Bootstrap 95% CI test revealing that there is much more 
significance to the Model's superiority compared to the Baseline in this 
scenario. This is due to the decrease in variance that comes with predicting 
the best streamer of the week vs. the best streamer of the day.
        Further testing, such as the weekly total accuracy, reveals that the
Model is superior to the Baseline at predicting how many fantasy points a player
will generate in a given week. The Model predicts the fantasy points of a
player-week on average 1.2 fantasy points closer than the Baseline, with a 55.5%
win rate. The significance tests reveal that this is a small effect, but
consistently statistically significant.
        The last test (Spearman) evaluates how well the Model can rank the whole
streamer pool for a given week compared to the Baseline. The results show that
the Model predicts the ranked order of available streamers better than the 
Baseline in 100% of the weeks. The Cohen's d reveals a huge effect size and the
significance tests demonstrate that the Model is consistently superior to the
Baseline at this application of ranking the whole streamer pool correctly.

## (4.4) Ranking Quality Evaluation

The final application of this Model that is tested is the Model's ability to
rank streamers from most fantasy points to least fantasy points for a given day,
using the the Model's fantasy points predictions. In these tests, the Model
excels and is much better than the Baseline.

In [18]:
# Evaluates the Model's ability to rank all the streamers in the correct order
# for a given day. Uses each day's Spearman correlation between predicted and
# actual fantasy points across all players on a daily slate.
def evaluate_rank_correlation(test_df, model, features, baseline_feature='FP_ROLLING_10', min_slate_size=5):
        test_df = test_df.copy()
        test_df['ML_PREDICTION'] = test_df['FP_ROLLING_10'] + model.predict(test_df[features])

        daily_corrs = []
        for date, slate in test_df.groupby('GAME_DATE'):
                if len(slate) < min_slate_size:
                        continue

                ml_corr, _ = stats.spearmanr(slate['ML_PREDICTION'], slate['FANTASY_PTS'])
                baseline_corr, _ = stats.spearmanr(slate[baseline_feature], slate['FANTASY_PTS'])

                if np.isnan(ml_corr) or np.isnan(baseline_corr):
                        continue

                daily_corrs.append({'GAME_DATE': date, 'SLATE_SIZE': len(slate),
                                'ML_CORR': ml_corr, 'BASELINE_CORR': baseline_corr,
                           })

        corr_df = pd.DataFrame(daily_corrs)
        corr_df['CORR_DIFF'] = corr_df['ML_CORR'] - corr_df['BASELINE_CORR']

        print("\n--- Full-Slate Rank Correlation (Spearman) ---")
        print(f"Days evaluated: {len(corr_df)} of 164")
        print(f"Mean ML correlation:       {corr_df['ML_CORR'].mean():.4f}")
        print(f"Mean Baseline correlation: {corr_df['BASELINE_CORR'].mean():.4f}")

        _paired_summary(corr_df['CORR_DIFF'].to_numpy(), "ML vs Baseline daily rank correlation")

        return corr_df


In [19]:
corr_df = evaluate_rank_correlation(test_df, model, features)
print()


--- Full-Slate Rank Correlation (Spearman) ---
Days evaluated: 164 of 164
Mean ML correlation:       0.5191
Mean Baseline correlation: 0.4604

ML vs Baseline daily rank correlation  (n = 164)
  Mean edge:            0.06 pts  (std: 0.05)
  ML win rate:          90.9%
  Cohen's d:            1.156
  Paired t-test:        t = 14.805, p = 0.0000
  Wilcoxon signed-rank: W = 575.0, p = 0.0000
  Bootstrap 95% CI:     [0.05, 0.07] pts



This output reveals that the Model ranks the whole available streamer pool
BETTER than the Baseline on 90.9% of the 164 test days. The Cohen's d reveals a
large effect size and the p-value demonstrates that this is statistically
significant.

In [20]:
# This is a similar test to the previous, except the evaluation is confined to
# only how the Model ranked the top-12 streamers of a given day. The top-12
# streamers can be selected in hindsight with the ACTUAL top-12 streamers of the
# day, or they can be selected by the Baseline's prediction of the top-12
# streamers of the day (using their 10-game fantasy point averages).
def evaluate_topk_rank_correlation(test_df, model, features, select_by, k=12,
                                   baseline_feature='FP_ROLLING_10',
                                   min_slate_size=5):
        if select_by not in ('actual', 'baseline'):
                raise ValueError("select_by must be 'actual' or 'baseline'")

        test_df = test_df.copy()
        test_df['ML_PREDICTION'] = test_df['FP_ROLLING_10'] + model.predict(test_df[features])

        daily_corrs = []
        for date, slate in test_df.groupby('GAME_DATE'):
                if len(slate) < max(min_slate_size, k):
                        continue

                sort_col = 'FANTASY_PTS' if select_by == 'actual' else baseline_feature
                top_slate = slate.sort_values(sort_col, ascending=False).head(k)

                ml_corr, _ = stats.spearmanr(top_slate['ML_PREDICTION'], top_slate['FANTASY_PTS'])
                baseline_corr, _ = stats.spearmanr(top_slate[baseline_feature], top_slate['FANTASY_PTS'])

                if np.isnan(ml_corr) or np.isnan(baseline_corr):
                        continue

                daily_corrs.append({'GAME_DATE': date, 'ML_CORR': ml_corr,
                            'BASELINE_CORR': baseline_corr,})

        corr_df = pd.DataFrame(daily_corrs)
        corr_df['CORR_DIFF'] = corr_df['ML_CORR'] - corr_df['BASELINE_CORR']

        print(f"\n--- Top-{k} Rank Correlation (Spearman, subset selected by {select_by.upper()}) ---")
        print(f"Days evaluated: {len(corr_df)} of 164")
        print(f"Mean ML correlation:       {corr_df['ML_CORR'].mean():.4f}")
        print(f"Mean Baseline correlation: {corr_df['BASELINE_CORR'].mean():.4f}")

        _paired_summary(corr_df['CORR_DIFF'].to_numpy(),
                     f"ML vs Baseline top-{k} rank correlation")

        return corr_df


In [21]:
topk_actual_df = evaluate_topk_rank_correlation(test_df, model, features, select_by='actual', k=12)
print()


--- Top-12 Rank Correlation (Spearman, subset selected by ACTUAL) ---
Days evaluated: 164 of 164
Mean ML correlation:       0.1636
Mean Baseline correlation: 0.1303

ML vs Baseline top-12 rank correlation  (n = 164)
  Mean edge:            0.03 pts  (std: 0.21)
  ML win rate:          59.8%
  Cohen's d:            0.163
  Paired t-test:        t = 2.084, p = 0.0388
  Wilcoxon signed-rank: W = 5138.5, p = 0.0076
  Bootstrap 95% CI:     [0.00, 0.06] pts



This test is using the ACTUAL top-12 streamers for a given day, and evaluating
how the Model ranked these players against each other. The Model ranks these 
players better than the Baseline on 59.8% of the days. Both the Model's and the
Baseline's correlation scores dropped compared to the last test, as this is a
much harder task due to the variance of the top-12 streamers and not allowing
the model to rank the consistently bad players. Cohen's d reveals a small effect
size, but the p-value is under the .05 threshold, meaning that this is still a
statistically significant finding.

In [22]:
topk_baseline_df = evaluate_topk_rank_correlation(test_df, model, features, select_by='baseline', k=12)
print()


--- Top-12 Rank Correlation (Spearman, subset selected by BASELINE) ---
Days evaluated: 164 of 164
Mean ML correlation:       0.2036
Mean Baseline correlation: 0.0901

ML vs Baseline top-12 rank correlation  (n = 164)
  Mean edge:            0.11 pts  (std: 0.34)
  ML win rate:          57.9%
  Cohen's d:            0.331
  Paired t-test:        t = 4.243, p = 0.0000
  Wilcoxon signed-rank: W = 4353.0, p = 0.0002
  Bootstrap 95% CI:     [0.06, 0.17] pts



This test is using the top-12 streamers for a given day, selected by the
Baseline. This means that the top-12 streamers are those with the highest
fantasy point averages over the past 10-games from the available streamer pool.
The Model ranks these players better than the Baseline on 57.9% of the days.
The correlation score of the Baseline drops heavily, but the Model's score
remains good. The Cohen's d reveals a medium effect size and the p-value of 0
demonstrates that this is statistically significant. This is arguably the
model's BEST APPLICATION METRIC, and it is one of the most realistic scenarios
in which the model could be actually used in a NBA fantasy league. Having the
model rank the top streamers of the waiver wire can allow fantasy managers to
improve their decisions when choosing a streamer to pick up, providing much
more information than just going off of fantasy point averages or rostered 
percentage (which are the metrics which ESPN sorts available players by).

# (5) Conclusion and Next Steps

Due to the nature of predicting NBA player performances, there will always be
variance and noise. However, this model has proven itself to excel in predicting
the best streamer of a given week, ranking the whole streamer pool for a given
day, and ranking specifically the Baseline selected top-12 streamers for a given
day. These 3 areas demonstrate the capability of the model to be used in actual
NBA fantasy leagues to help managers filter out bad streamer options and
eventually make the most informed decision possible when selecting a streamer.
The model can be improved by increasing its prediction accuracy, specifically,
in the best streamer of the day test. This could be done by adding a feature
that includes betting or Vegas odds, allowing the model to better predict if a
game will be a blowout or not. However, due to the model's proven ability in
certain areas, I believe that the model is ready to be deployed and used in
actual fantasy leagues. The next steps are to design a pipeline to feed the
model the necessary data to make a prediction live, when a league is under way.
The model should be continually retrained as each NBA season goes on, to give
the model new clarity on new trends in the NBA and also more data in general
to improve accuracy.

The model's prediction should always be taken as advice instead of a concrete 
decision, as managers should always review the Model's picks and use their own
decision-making to choose the best streamer. The Model is designed to be a tool
to HELP fantasy managers, not make the choice for them.

In [24]:
# Saving the model for later use
import joblib

MODEL_DIRECTORY = "saved_models"
os.makedirs(MODEL_DIRECTORY, exist_ok=True)

file_path = os.path.join(MODEL_DIRECTORY, "nba_streamer_model.pkl")

joblib.dump(model, file_path)
print("NBA Streamer Prediction Model Saved")



NBA Streamer Prediction Model Saved
